# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.15 — FAST
## Schwarzschild Classical Vacuum Benchmark Audit

On teste le candidat exact :

\[
ds^2=-f(r)dt^2+\frac{dr^2}{f(r)}+r^2d\Omega^2,
\qquad f(r)=1-\frac{2M}{r},
\]

dans la région \(r>2M\), avec le vecteur GVH statique aligné sur la normale ADM :

\[
s=1,\qquad v_i=0,\qquad u^\mu=n^\mu.
\]

Le notebook **ne suppose pas** que ce candidat doit réussir. Il vérifie d'abord la géométrie Schwarzschild GR, puis teste l'équation de lapse GVH. Un échec du vecteur aligné ne réfute pas un ansatz radial plus général \(u^r(r)\neq0\).

In [1]:
# SCH15.1 — Environment and upstream gate
from __future__ import annotations
import sympy as sp, json, sys
from pathlib import Path

UPSTREAM = {
 "p3314": {
   "canonical_user_executed_sha256": "ad810c82515ea7cae7462894a07351936644e760ebc2ea97f965182302d8e886",
   "canonical_user_executed_size_bytes": 39056,
   "WEAK_FIELD_BENCHMARK_PASS_GENERIC_STABLE_SUBBRANCH": True,
   "WEAK_FIELD_BENCHMARK_PASS_ALL_COUPLINGS": False,
   "SCHWARZSCHILD_BENCHMARK_AUTHORIZED": True,
   "CLASSICAL_PREDICTIONS_AUTHORIZED": False
 }
}
UPSTREAM_GATE = (
    UPSTREAM["p3314"]["WEAK_FIELD_BENCHMARK_PASS_GENERIC_STABLE_SUBBRANCH"]
    and not UPSTREAM["p3314"]["WEAK_FIELD_BENCHMARK_PASS_ALL_COUPLINGS"]
    and UPSTREAM["p3314"]["SCHWARZSCHILD_BENCHMARK_AUTHORIZED"]
    and not UPSTREAM["p3314"]["CLASSICAL_PREDICTIONS_AUTHORIZED"]
)
assert UPSTREAM_GATE
print("Python =", sys.version.split()[0])
print("SymPy =", sp.__version__)
print("UPSTREAM_GATE =", UPSTREAM_GATE)
print("P3314_CANONICAL_SHA256 =", UPSTREAM["p3314"]["canonical_user_executed_sha256"])

Python = 3.13.15
SymPy = 1.14.0
UPSTREAM_GATE = True
P3314_CANONICAL_SHA256 = ad810c82515ea7cae7462894a07351936644e760ebc2ea97f965182302d8e886


## 1. Schwarzschild est bien vide en GR

On vérifie directement :

\[
R_{\mu\nu}=0.
\]

Sur les tranches statiques :

\[
N=\sqrt f,\qquad N^i=0,\qquad K_{ij}=0,
\]

et l'on vérifie aussi :

\[
{}^{(3)}R=0.
\]

In [2]:
# SCH15.2 — 4D Schwarzschild Ricci tensor
t,r,th,ph,M = sp.symbols("t r theta phi M", positive=True, real=True)
x4=(t,r,th,ph)
f=sp.simplify(1-2*M/r)
g=sp.diag(-f,1/f,r**2,r**2*sp.sin(th)**2)
gi=sp.simplify(g.inv())

G4=[[[sp.simplify(sp.Rational(1,2)*sum(
    gi[a,d]*(sp.diff(g[d,c],x4[b])+sp.diff(g[d,b],x4[c])-sp.diff(g[b,c],x4[d]))
    for d in range(4)))
    for c in range(4)] for b in range(4)] for a in range(4)]

R4=sp.zeros(4,4)
for a in range(4):
    for b in range(4):
        e=0
        for c in range(4):
            e += sp.diff(G4[c][a][b],x4[c]) - sp.diff(G4[c][a][c],x4[b])
            for d in range(4):
                e += G4[c][c][d]*G4[d][a][b]-G4[c][b][d]*G4[d][a][c]
        R4[a,b]=sp.factor(sp.simplify(e))

SCHWARZSCHILD_4D_RICCI_ZERO_PASS = all(R4[i,j]==0 for i in range(4) for j in range(4))
assert SCHWARZSCHILD_4D_RICCI_ZERO_PASS
print("f(r) =", f)
print("SCHWARZSCHILD_4D_RICCI_ZERO_PASS =", SCHWARZSCHILD_4D_RICCI_ZERO_PASS)

f(r) = (-2*M + r)/r
SCHWARZSCHILD_4D_RICCI_ZERO_PASS = True


In [3]:
# SCH15.3 — 3D ADM scalar curvature
x3=(r,th,ph)
h=sp.diag(1/f,r**2,r**2*sp.sin(th)**2)
hi=sp.simplify(h.inv())

G3=[[[sp.simplify(sp.Rational(1,2)*sum(
    hi[a,d]*(sp.diff(h[d,c],x3[b])+sp.diff(h[d,b],x3[c])-sp.diff(h[b,c],x3[d]))
    for d in range(3)))
    for c in range(3)] for b in range(3)] for a in range(3)]

R3ij=sp.zeros(3,3)
for a in range(3):
    for b in range(3):
        e=0
        for c in range(3):
            e += sp.diff(G3[c][a][b],x3[c])-sp.diff(G3[c][a][c],x3[b])
            for d in range(3):
                e += G3[c][c][d]*G3[d][a][b]-G3[c][b][d]*G3[d][a][c]
        R3ij[a,b]=sp.factor(sp.simplify(e))

R3=sp.factor(sp.simplify(sum(hi[i,j]*R3ij[i,j] for i in range(3) for j in range(3))))
Kij_ZERO=True
SCHWARZSCHILD_R3_ZERO_PASS=(R3==0)
SCHWARZSCHILD_GR_ADM_VACUUM_CONSTRAINT_PASS=Kij_ZERO and SCHWARZSCHILD_R3_ZERO_PASS
assert SCHWARZSCHILD_GR_ADM_VACUUM_CONSTRAINT_PASS
print("R3 =", R3)
print("Kij_ZERO =", Kij_ZERO)
print("SCHWARZSCHILD_GR_ADM_VACUUM_CONSTRAINT_PASS =", SCHWARZSCHILD_GR_ADM_VACUUM_CONSTRAINT_PASS)

R3 = 0
Kij_ZERO = True
SCHWARZSCHILD_GR_ADM_VACUUM_CONSTRAINT_PASS = True


## 2. Blocs GVH pour le vecteur statique aligné

Avec

\[
s=1,\quad v_i=0,\quad K_{ij}=0,
\]

on obtient :

\[
A=0,\qquad C_i=0,\qquad D_{ij}=0,
\]

mais

\[
B_i=a_i^{(n)}=D_i\ln N.
\]

Pour Schwarzschild :

\[
a_r=\partial_r\ln\sqrt f
=\frac{M}{r(r-2M)}.
\]

Donc :

\[
a^2=\frac{M^2}{r^3(r-2M)}.
\]

Le secteur vectoriel devient :

\[
\boxed{\mathcal L_u=(c_1+c_4)a^2}.
\]

In [4]:
# SCH15.4 — Aligned GVH vector invariants
N=sp.sqrt(f)
ar=sp.factor(sp.diff(sp.log(N),r))
a2=sp.factor(hi[0,0]*ar**2)

c1,c2,c3,c4=sp.symbols("c1 c2 c3 c4", real=True)
c14=sp.factor(c1+c4)

I1=sp.factor(-a2)
theta=sp.Integer(0)
I3=sp.Integer(0)
Lu=sp.factor(-c1*I1-c2*theta**2-c3*I3+c4*a2)

ALIGNED_VECTOR_NORM_PASS=True
ALIGNED_VECTOR_BLOCKS_MATERIALIZED=True
ALIGNED_VECTOR_LAGRANGIAN_CROSSCHECK_PASS=(sp.simplify(Lu-c14*a2)==0)
assert ALIGNED_VECTOR_LAGRANGIAN_CROSSCHECK_PASS

print("a_r =", ar)
print("a^2 =", a2)
print("Lu =", Lu)
print("ALIGNED_VECTOR_LAGRANGIAN_CROSSCHECK_PASS =", ALIGNED_VECTOR_LAGRANGIAN_CROSSCHECK_PASS)

a_r = M/(r*(-2*M + r))
a^2 = M**2/(r**3*(-2*M + r))
Lu = M**2*(c1 + c4)/(r**3*(-2*M + r))
ALIGNED_VECTOR_LAGRANGIAN_CROSSCHECK_PASS = True


## 3. Test décisif : équation de lapse

Pour éviter de conclure seulement à partir de \(\mathcal L_u\neq0\), on fait varier explicitement le lapse.

Avec la métrique spatiale Schwarzschild maintenue fixe et un lapse radial général \(X(r)\), la densité radiale vectorielle est :

\[
L_{\rm vec}^{\rm rad}
=
\sqrt h\,(c_1+c_4)\,h^{rr}\frac{X'^2}{X}.
\]

On calcule :

\[
\mathcal E_X=
\frac{\partial L}{\partial X}
-\frac{d}{dr}\frac{\partial L}{\partial X'}.
\]

Puis on évalue \(X=\sqrt f\).

In [5]:
# SCH15.5 — Exact lapse Euler-Lagrange residual
X=sp.Function("X")(r)
sqrt_h_noang=sp.factor(r**2/sp.sqrt(f))
hrr=f

Lrad=sp.factor(sqrt_h_noang*c14*hrr*sp.diff(X,r)**2/X)
EL=sp.factor(sp.diff(Lrad,X)-sp.diff(sp.diff(Lrad,sp.diff(X,r)),r))
EL_Sch=sp.factor(sp.simplify(EL.subs(X,N).doit()))

EL_expected=sp.factor(c14*M**2/(sp.sqrt(r)*(r-2*M)**sp.Rational(3,2)))
LAPSE_RESIDUAL_EXACT_CROSSCHECK_PASS=(sp.simplify(EL_Sch-EL_expected)==0)
assert LAPSE_RESIDUAL_EXACT_CROSSCHECK_PASS

print("lapse_EL_on_Schwarzschild =", EL_Sch)
print("LAPSE_RESIDUAL_EXACT_CROSSCHECK_PASS =", LAPSE_RESIDUAL_EXACT_CROSSCHECK_PASS)

lapse_EL_on_Schwarzschild = M**2*(c1 + c4)/(sqrt(r)*(-2*M + r)**(3/2))
LAPSE_RESIDUAL_EXACT_CROSSCHECK_PASS = True


Dans la région extérieure \(r>2M\), \(M>0\),

\[
\frac{M^2}{\sqrt r\,(r-2M)^{3/2}}>0.
\]

Ainsi l'équation de lapse impose nécessairement :

\[
\boxed{c_1+c_4=0}
\]

pour ce candidat statique aligné.

Or le benchmark faible champ stable avait :

\[
K_V=c_1+c_4>0.
\]

Les deux conditions sont incompatibles.

In [6]:
# SCH15.6 — Stable-branch incompatibility
EL_over_c14=sp.factor(sp.simplify(EL_Sch/c14))
witness_coeff=sp.simplify(EL_over_c14.subs({M:1,r:4}))

LAPSE_RESIDUAL_PROPORTIONAL_TO_C14=(sp.simplify(EL_Sch-c14*EL_over_c14)==0)
LAPSE_RESIDUAL_COEFFICIENT_NONZERO=(witness_coeff!=0)
STATIC_ALIGNED_LAPSE_EOM_REQUIRES_C14_ZERO=(
    LAPSE_RESIDUAL_PROPORTIONAL_TO_C14 and LAPSE_RESIDUAL_COEFFICIENT_NONZERO
)

WEAK_FIELD_VECTOR_NO_GHOST_REQUIRES_C14_POSITIVE=True
STATIC_ALIGNED_SCHWARZSCHILD_COMPATIBLE_WITH_STABLE_WEAK_FIELD_BRANCH=False

STATIC_ALIGNED_SCHWARZSCHILD_CANDIDATE_REFUTED_ON_STABLE_BRANCH=all([
    STATIC_ALIGNED_LAPSE_EOM_REQUIRES_C14_ZERO,
    WEAK_FIELD_VECTOR_NO_GHOST_REQUIRES_C14_POSITIVE,
    not STATIC_ALIGNED_SCHWARZSCHILD_COMPATIBLE_WITH_STABLE_WEAK_FIELD_BRANCH,
])
assert STATIC_ALIGNED_SCHWARZSCHILD_CANDIDATE_REFUTED_ON_STABLE_BRANCH

print("EL/(c1+c4) =", EL_over_c14)
print("coefficient witness M=1,r=4 =", witness_coeff)
print("STATIC_ALIGNED_LAPSE_EOM_REQUIRES_C14_ZERO =", STATIC_ALIGNED_LAPSE_EOM_REQUIRES_C14_ZERO)
print("STATIC_ALIGNED_SCHWARZSCHILD_CANDIDATE_REFUTED_ON_STABLE_BRANCH =", STATIC_ALIGNED_SCHWARZSCHILD_CANDIDATE_REFUTED_ON_STABLE_BRANCH)

EL/(c1+c4) = M**2/(sqrt(r)*(-2*M + r)**(3/2))
coefficient witness M=1,r=4 = sqrt(2)/8
STATIC_ALIGNED_LAPSE_EOM_REQUIRES_C14_ZERO = True
STATIC_ALIGNED_SCHWARZSCHILD_CANDIDATE_REFUTED_ON_STABLE_BRANCH = True


## 4. Portée du résultat

Ce résultat ne montre pas que Schwarzschild est impossible dans GVH.

Il exclut seulement, sur la branche faible champ stable déjà retenue, le candidat :

\[
u^\mu=n^\mu
\]

strictement statique.

Il reste à tester le champ sphériquement symétrique général :

\[
u^\mu=(u^t(r),u^r(r),0,0),
\]

avec :

\[
g_{\mu\nu}u^\mu u^\nu=-1.
\]

Un \(u^r(r)\neq0\) modifie les blocs \(A,B,C,D\) et peut changer entièrement l'équation de lapse.

In [7]:
# SCH15.7 — Final scientific classifier
SCHWARZSCHILD_GR_GEOMETRY_VERIFIED=all([
    SCHWARZSCHILD_4D_RICCI_ZERO_PASS,
    SCHWARZSCHILD_GR_ADM_VACUUM_CONSTRAINT_PASS
])

SCHWARZSCHILD_STATIC_ALIGNED_VECTOR_CANDIDATE_CLASSIFIED=all([
    ALIGNED_VECTOR_NORM_PASS,
    ALIGNED_VECTOR_BLOCKS_MATERIALIZED,
    ALIGNED_VECTOR_LAGRANGIAN_CROSSCHECK_PASS,
    LAPSE_RESIDUAL_EXACT_CROSSCHECK_PASS,
    STATIC_ALIGNED_LAPSE_EOM_REQUIRES_C14_ZERO,
    STATIC_ALIGNED_SCHWARZSCHILD_CANDIDATE_REFUTED_ON_STABLE_BRANCH
])

SCHWARZSCHILD_BENCHMARK_PASS=False
SCHWARZSCHILD_BENCHMARK_STATUS="STATIC_ALIGNED_VECTOR_CANDIDATE_REFUTED_ON_STABLE_WEAK_FIELD_BRANCH_GENERAL_RADIAL_VECTOR_PENDING"

KERR_BENCHMARK_AUTHORIZED=False
CLASSICAL_PREDICTIONS_AUTHORIZED=False
QUANTIZATION_READY=False

SCH15_OBSTRUCTIONS=[
    "GENERAL-SPHERICALLY-SYMMETRIC-RADIAL-GVH-VECTOR-NOT-YET-CLASSIFIED",
    "FULL-SCHWARZSCHILD-GVH-FIELD-EQUATIONS-NOT-YET-CLOSED"
]

SCH15_LOCAL_AUDIT_PASS=all([
    UPSTREAM_GATE,
    SCHWARZSCHILD_GR_GEOMETRY_VERIFIED,
    SCHWARZSCHILD_STATIC_ALIGNED_VECTOR_CANDIDATE_CLASSIFIED,
    not SCHWARZSCHILD_BENCHMARK_PASS,
    not KERR_BENCHMARK_AUTHORIZED,
    not CLASSICAL_PREDICTIONS_AUTHORIZED,
    not QUANTIZATION_READY
])

SCH15_NEXT_AUTHORIZED=(
    "AUDIT-GENERAL-SPHERICALLY-SYMMETRIC-SCHWARZSCHILD-RADIAL-VECTOR-ANSATZ"
    if SCH15_LOCAL_AUDIT_PASS
    else "REPAIR-.3.3.15-SCHWARZSCHILD-STATIC-ALIGNED-AUDIT"
)
assert SCH15_LOCAL_AUDIT_PASS

print("SCHWARZSCHILD_GR_GEOMETRY_VERIFIED =", SCHWARZSCHILD_GR_GEOMETRY_VERIFIED)
print("SCHWARZSCHILD_STATIC_ALIGNED_VECTOR_CANDIDATE_CLASSIFIED =", SCHWARZSCHILD_STATIC_ALIGNED_VECTOR_CANDIDATE_CLASSIFIED)
print("SCHWARZSCHILD_BENCHMARK_PASS =", SCHWARZSCHILD_BENCHMARK_PASS)
print("SCHWARZSCHILD_BENCHMARK_STATUS =", SCHWARZSCHILD_BENCHMARK_STATUS)
print("KERR_BENCHMARK_AUTHORIZED =", KERR_BENCHMARK_AUTHORIZED)
print("SCH15_OBSTRUCTIONS =", SCH15_OBSTRUCTIONS)
print("SCH15_NEXT_AUTHORIZED =", SCH15_NEXT_AUTHORIZED)

SCHWARZSCHILD_GR_GEOMETRY_VERIFIED = True
SCHWARZSCHILD_STATIC_ALIGNED_VECTOR_CANDIDATE_CLASSIFIED = True
SCHWARZSCHILD_BENCHMARK_PASS = False
SCHWARZSCHILD_BENCHMARK_STATUS = STATIC_ALIGNED_VECTOR_CANDIDATE_REFUTED_ON_STABLE_WEAK_FIELD_BRANCH_GENERAL_RADIAL_VECTOR_PENDING
KERR_BENCHMARK_AUTHORIZED = False
SCH15_OBSTRUCTIONS = ['GENERAL-SPHERICALLY-SYMMETRIC-RADIAL-GVH-VECTOR-NOT-YET-CLASSIFIED', 'FULL-SCHWARZSCHILD-GVH-FIELD-EQUATIONS-NOT-YET-CLOSED']
SCH15_NEXT_AUTHORIZED = AUDIT-GENERAL-SPHERICALLY-SYMMETRIC-SCHWARZSCHILD-RADIAL-VECTOR-ANSATZ


In [8]:
# SCH15.8 — Machine-readable artifact
artifact={
 "notebook":"GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.15_Schwarzschild_Classical_Vacuum_Benchmark_Audit_FAST",
 "execution_scope":"SCHWARZSCHILD_STATIC_ALIGNED_GVH_VECTOR_CANDIDATE",
 "upstream":UPSTREAM,
 "gr_geometry":{
   "ricci4_zero":SCHWARZSCHILD_4D_RICCI_ZERO_PASS,
   "R3_zero":SCHWARZSCHILD_R3_ZERO_PASS,
   "Kij_zero":Kij_ZERO
 },
 "aligned_vector":{
   "a_r":str(ar),"a_squared":str(a2),"L_u":str(Lu),
   "lagrangian_crosscheck_pass":ALIGNED_VECTOR_LAGRANGIAN_CROSSCHECK_PASS
 },
 "lapse_obstruction":{
   "EL_residual":str(EL_Sch),
   "EL_over_c1_plus_c4":str(EL_over_c14),
   "requires_c1_plus_c4_zero":STATIC_ALIGNED_LAPSE_EOM_REQUIRES_C14_ZERO
 },
 "branch_compatibility":{
   "weak_field_requires":"c1+c4>0",
   "static_aligned_requires":"c1+c4=0",
   "compatible":STATIC_ALIGNED_SCHWARZSCHILD_COMPATIBLE_WITH_STABLE_WEAK_FIELD_BRANCH
 },
 "scientific_status":{
   "SCHWARZSCHILD_GR_GEOMETRY_VERIFIED":SCHWARZSCHILD_GR_GEOMETRY_VERIFIED,
   "SCHWARZSCHILD_STATIC_ALIGNED_VECTOR_CANDIDATE_CLASSIFIED":SCHWARZSCHILD_STATIC_ALIGNED_VECTOR_CANDIDATE_CLASSIFIED,
   "SCHWARZSCHILD_BENCHMARK_PASS":SCHWARZSCHILD_BENCHMARK_PASS,
   "SCHWARZSCHILD_BENCHMARK_STATUS":SCHWARZSCHILD_BENCHMARK_STATUS,
   "KERR_BENCHMARK_AUTHORIZED":KERR_BENCHMARK_AUTHORIZED
 },
 "verdict":{"SCH15_LOCAL_AUDIT_PASS":SCH15_LOCAL_AUDIT_PASS,"obstructions":SCH15_OBSTRUCTIONS},
 "next_authorized":SCH15_NEXT_AUTHORIZED,
 "scope_note":"Static aligned Schwarzschild GVH-vector candidate refuted on stable weak-field branch; general radial vector remains open."
}
export_dir=Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True,exist_ok=True)
artifact_path=export_dir/"gvh_0.3.2.7.3.7.3.3.15_Schwarzschild_Classical_Vacuum_Benchmark_Audit_FAST.json"
artifact_path.write_text(json.dumps(artifact,indent=2,ensure_ascii=False),encoding="utf-8")
print("SCH15 artifact =",artifact_path)

SCH15 artifact = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.15_Schwarzschild_Classical_Vacuum_Benchmark_Audit_FAST.json


# Conclusion

`.3.3.15` est un diagnostic ciblé :

\[
\boxed{
\text{Schwarzschild GR} + u^\mu=n^\mu
\Rightarrow c_1+c_4=0
}
\]

alors que la branche faible champ stable impose :

\[
c_1+c_4>0.
\]

Le candidat statique aligné est donc réfuté sur cette branche.

Le prochain audit doit autoriser :

\[
\boxed{u^r(r)\neq0}.
\]